# Test chFPGA Firmware for t0.CRS Platform, Pocket Correlator Mode

This notebook tests all data products corresponding to each stage of the Pocket Correlator firmware, beginning with ADC data, working through the different modules of the Channelizer, and finally the Correlator. This notebook combines real data with custom test patterns for verification.

Revisions:
- **March 5, 2025**: initial notebook, with data taken using the Deep Dish Development Array (D3A). There are several debugging to-do's that remain, which correspond to some issues in the Correlator.
- **June 11, 2025**: added some new updates to this (and others) test notebook; note that the correlator is not working at this time, so any plots in the correlator section are either old or not a reference for a functioning correlator.
- **October 13, 2025**: includes changes to accomodate a big UCORR update, allowing for software integration methods that have been unavailable since June 2025. Also includes updates for double-resolution FFT capture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 10})
import os
import time

# Program and Connect to the CRS Board

Remember to change the path to **corriscope** listed in the cell below to match its location on your machine, as well as the **serial** of the board to which you are connecting.

In [ ]:
run ../../../corriscope/pocket_correlator.py --serial 0050 --stderr_log_level debug --prog 2

# Capture ADC Data

In [ ]:
# Set the capture mode to 0 so each ADC input can be verified.
pc.set_ucap_mode(mode = 0)

In [ ]:
n_adc_frames = 2*500 # must be multiple of 2 in UCAP mode 0; more frames takes longer to capture!
n_adc_frames

In [ ]:
d_adc = pc.read_adc_frames(n_adc_frames)

In [ ]:
d_adc.shape

In [ ]:
adc_rms = np.array([np.std(d_adc[n].flatten()) for n in range(pc.NPOLS)])
power_adc = 10*np.log10((adc_rms*(1/2**14))**2/100/1e-3)
power_adc

In [ ]:
print('ADC Overvoltage =', pc.i.GPIO.ADC_OV)
print('ADC Overrange =', pc.i.GPIO.ADC_OR)

In [ ]:
d_adc_psd = np.mean(np.abs(np.fft.rfft(d_adc, axis = 2)), axis = 1)**2

In [ ]:
plt.figure(figsize  = (12, 18))

for n in range(pc.NPOLS):

    plt.subplot(pc.NPOLS, 2, 2*n + 1)
    plt.title(f'ADC{n + 1}, {np.round(power_adc[n], 1)} dBm at ADC')
    plt.plot(np.arange(2**14), d_adc[n, 0], color = 'blue') #, marker = '.', s = 25)
    plt.xlim(0, 256)
    plt.ylabel('$V$ [LSBs]')
    plt.xlabel('sample')
    plt.grid()
    
    plt.subplot(pc.NPOLS, 2, 2*n + 2)
    plt.title(f'ADC{n + 1}, NumPy FFT')
    plt.semilogy(pc.f, d_adc_psd[n][:-1], color = 'black')
    plt.ylabel('power [arb.]')
    plt.xlabel('frequency [MHz]')
    plt.xlim(0, 1600)
    plt.grid()

plt.tight_layout()

# Capture FFT Data

In [ ]:
n_fft_frames = n_adc_frames
n_fft_frames

## Capture All FFT bins, Most Significant (Upper) 16+16 Bits
Capture the upper 16+16 bits of the FFT data. Note that this method is faster than recovering the full 32+32-bit resolution FFT data, but depending on the signal, the upper 16+16 bits may recover the signal well, if it is mostly contained in the upper 16+16 bits, or be heavily quantized if it is not largely contained in this bit range.

In [ ]:
t1 = time.time()
d_fft_16_16 = pc.read_fft_frames(n_frames = n_fft_frames, fft_bit_res = '16+16', data_source = 'adc')
t2 = time.time()

In [ ]:
print(f'Took {t2 - t1} s to capture {n_fft_frames} FFT frames at 16+16-bit resolution.')

In [ ]:
d_fft_16_16.shape

In [ ]:
d_fft_16_16_mean_mag = np.mean(np.abs(d_fft_16_16), axis = 1)

In [ ]:
plt.figure(figsize = (12, 10))
plt.suptitle('raw FFT, 16+16 bits')

for n in range(pc.NPOLS):

    plt.subplot(8, 3, 3*n + 1)
    if n == 0:
        plt.title('real')
    plt.plot(pc.f, d_fft_16_16[n, 0].real)
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.ylabel(f'ADC{n + 1}')
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 2)
    if n == 0:
        plt.title('imaginary')
    plt.plot(pc.f, d_fft_16_16[n, 0].imag, color = 'C1')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 3)
    if n == 0:
        plt.title('power')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft_16_16[n])**2, axis = 0), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

## Capture FFT Data, 32+32 Bit Resolution

In [ ]:
t1 = time.time()
d_fft = pc.read_fft_frames(n_frames = n_fft_frames, 
                           fft_bit_res = '32+32', 
                           data_source = 'adc', 
                           period = None,
                           verbose = 0
                           )
t2 = time.time()

In [ ]:
print(f'Took {t2 - t1} s to capture {n_fft_frames} FFT frames at full 32+32-bit resolution.')

In [ ]:
d_fft_mean_mag = np.mean(np.abs(d_fft), axis = 1)

In [ ]:
plt.figure(figsize = (12, 10))
plt.suptitle('raw FFT, 32+32 bits')

for n in range(pc.NPOLS):

    plt.subplot(8, 3, 3*n + 1)
    if n == 0:
        plt.title('real')
    plt.plot(pc.f, d_fft[n, 0].real)
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    # plt.ylim(-1e6, 1e6)
    plt.ylabel(f'ADC{n + 1}')
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 2)
    if n == 0:
        plt.title('imaginary')
    plt.plot(pc.f, d_fft[n, 0].imag, color = 'C1')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    # plt.ylim(-1e6, 1e6)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 3)
    if n == 0:
        plt.title('power')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft[n])**2, axis = 0), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

In [ ]:
f_bin = np.arange(1024 - 512, 8191 + 1024, 1024)[:-1]
n_hist_bins = 128
hist_bins = np.arange(n_hist_bins)
hist_pol_sel = 0

plt.figure(figsize = (12, 8))
plt.suptitle(f'histograms of firmware FFT bins, ADC{hist_pol_sel + 1}')

for n in range(len(f_bin)):

    # c_fft_re, bin_edges = np.histogram(a >> 30, bins = n_bins) # np.arange(-256, 255 + 1)) # , bins = np.arange(-2**20, 2**20 + 1, 1))
    # c_fft_im, bin_edges = np.histogram(b >> 30, bins = n_bins) #np.arange(-256, 255 + 1)) # , bins = np.arange(-2**20, 2**20 + 1, 1))
    c_fft_re, bin_edges = np.histogram(d_fft[hist_pol_sel, :, f_bin[n]].real, bins = n_hist_bins)
    c_fft_im, bin_edges = np.histogram(d_fft[hist_pol_sel, :, f_bin[n]].imag, bins = n_hist_bins) 
    # bin_cents = np.arange(-2**20, 2**20)
    # bin_cents = bin_edges
    
    # plt.figure(figsize = (12, 4))
    # plt.suptitle('')
    
    plt.subplot(4, 2, n + 1)
    plt.title(f'{pc.f[f_bin[n]]} MHz')
    plt.plot(hist_bins, c_fft_re/np.sum(c_fft_re))
    plt.plot(hist_bins, c_fft_im/np.sum(c_fft_im))
    # plt.vlines(-8, 0, np.max(c_test_re/np.sum(c_test_re)), color = 'black', linestyle = '--')
    # plt.vlines(8, 0, np.max(c_test_re/np.sum(c_test_re)), color = 'black', linestyle = '--')
    # plt.xticks(bin_cents)

    if n in [0, 2, 4, 6]:
        plt.ylabel('normalized \ncounts')
    if n in [6, 7]:
            plt.xlabel('histogram bin [arb.]')
    plt.grid()
    
plt.tight_layout()

## Compare the 16+16-Bit and 32+32-Bit FFTs

In [ ]:
plt.figure(figsize = (12, 12))
plt.suptitle('FFT bit-depth comparison')

for n in range(pc.NPOLS):

    plt.subplot(8, 2, 2*n + 1)
    if n == 0:
        plt.title('16+16 bits')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft_16_16[n])**2, axis = 0), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.ylabel(f'ADC{n + 1}')
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 2, 2*n + 2)
    if n == 0:
        plt.title('32+32 bits')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft[n])**2, axis = 0), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

## Plot the Raw ADC Data, NumPy FFT of ADC Data, and Full 32+32-Bit Firmware FFT

In [ ]:
plt.figure(figsize = (16, 16))
plt.suptitle('numpy and firmware FFT comparison')

for n in range(pc.NPOLS):

    plt.subplot(8, 3, 3*n + 1)
    if n == 0:
        plt.title('ADC data')
    plt.plot(np.arange(2**14), d_adc[n, 0]) #, marker = '.', s = 25)
    plt.hlines(8191, 0, 128, color = 'black', linestyle = '--')
    plt.hlines(-8192, 0, 128, color = 'black', linestyle = '--')
    plt.ylim(-256, 255)
    # plt.yticks([-8192, -4096, -2048, 2047, 4095, 8181])
    plt.yticks([-256, -128, -64, 63, 127, 255])
    plt.xlim(0, 256)
    plt.ylabel(f'ADC{n + 1}')
    plt.xlabel('sample')
    plt.grid()
    
    if n + 1 == 8:
        plt.xlabel('sample')
        
    plt.subplot(8, 3, 3*n + 2)
    if n == 0:
        plt.title('numpy FFT of ADC data')
    plt.semilogy(pc.f, d_adc_psd[n][:-1], color = 'C1')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')
    
    plt.subplot(8, 3, 3*n + 3)
    if n == 0:
        plt.title('firmware FFT, 32+32 bits')
    plt.semilogy(pc.f, np.mean(np.abs(d_fft[n])**2, axis = 0), color = 'C2')
    plt.xticks(np.arange(0, 1600 + 400, 400))
    plt.xlim(0, 1600)
    plt.grid()

    if n + 1 == 8:
        plt.xlabel('frequency [MHz]')

plt.tight_layout()

# Compute Floating-Point Digital Gains

With the 32+32-bit FFT, the digital gains now multiply the FFT output to a 49+49-bit number, of which the upper 4+4 bits are preserved.

Also recall that the digital gains now support floating-point format, with 11 bits dedicated to the linear component and 5 bits dedicated to the log component per frequency bin. Also note that there is a "common" log component that applies to each channelizer, and is an additional 5-bit value that is *added* to the per-bin log component, for a total of 6 bits for the log component. Therefore, the maximum digital gain is $2^{11} \times 2^{2^6} = 2048 \times 2^{64}$. Each frequency bin now has a unique digital gain (i.e., the log component is no longer shared for a given analog channel).

## Test with FUNCGEN, Flat Spectrum

I will first test the new gains with a flat spectrum, to confirm that the values are found in the right place after being read from the Scaler.

By setting set_data_source('a', a = 1), I am generating a FUNCGEN signal whose value is 1 for a single frame, $M = 2^{14}$. The FUNCGEN generates 14-bit values, which are shifted by 18 bits when the FFT is bypassed. Even (odd) indices of the FUNCGEN become the real (imaginary) parts of the 32+32-bit output of the FFT when it is bypassed. 

Therefore, the configuration below, when interpreted as 32+32-bit FFT data, will be equivalent to $2^{18} + j~2^{18}$ for all frequency bins.

Another important subtelty to keep in mind is that the frequency bins of the channelizers are scrambled, such that the inverse bin map must be applied to any custom function written to the buffers of the FUNCGEN. However, applying the inverse bin map is not strictly necessary given that the data is captured in scrambled frequency bin-order. Therefore, if a custom function is written in a certain order in Python, it will be read back in the same order, as well as any downstream data products. 

As such, I will not apply a bin map anywhere in the "flat" section for simplicity. However, with frequency-dependent or real data, we will need to apply a bin map to make sense of the data.

In [ ]:
d_fft_flat = pc.read_fft_frames(n_frames = 2, 
                                fft_bit_res = '32+32', 
                                data_source = 'a', 
                                fft_bypass = True, 
                                period = None,
                                a = 1
                                )

In [ ]:
d_fft_flat_mean_mag = np.mean(np.abs(d_fft_flat), axis = 1)

In [ ]:
d_fft_flat.shape

In [ ]:
d_fft_flat[0,0,0]

In [ ]:
np.log2(262144)

In [ ]:
# Plot the flat data to verify that it's being set up and read correctly

plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.title('test flat FFT data')
plt.plot(pc.f, d_fft_flat[0, 0].real, label = 'measured')
plt.hlines(2**18, 0, 1600, color = 'red', linestyle = '--', label = 'expected')
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.subplot(312)
plt.plot(pc.f, d_fft_flat[0, 0].imag)
plt.hlines(2**18, 0, 1600, color = 'red', linestyle = '--')
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, d_fft_flat_mean_mag[0])
plt.hlines(2**18*np.sqrt(2), 0, 1600, color = 'red', linestyle = '--')
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()

plt.tight_layout()

### Set the Test Gains and Capture Scaler Data

Because we have set a single integer value as our signal, it is impossible for us to recover any value from the Scaler that is not an integer! There is no noise on the signal, so its output is deterministic. 

Therefore, the target magnitude must be set to be the quadrature sum of two integers For example, if I want the real and imaginary components to always be +1 (positive since we have set the spectrum to always be positive), then my Scaler magnitude must be

\begin{equation}
    |\text{Scaler}| = \sqrt{1^2 + 1^2} = \sqrt{2}.
\end{equation}

I earlier had the real and imaginary components as always +2, but this ends up saturating the correlator later on...

In [ ]:
test_target = np.sqrt(2) # The target magnitude as a 4-bit number
test_target_full = test_target*2**44 # The target magnitude as a 4-bit number on top of a 44-bit number 

In [ ]:
test_gains_flat = test_target_full/d_fft_flat_mean_mag

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.title('verification of gain computation')
plt.plot(pc.f, d_fft_flat_mean_mag[0])
plt.ylabel('|FFT|')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.plot(pc.f, test_gains_flat[0])
plt.ylabel('gains')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, d_fft_flat_mean_mag[0]*test_gains_flat[0]/2**44, label = 'gain $\\times$ |FFT|')
plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('simulated \nscaler magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

### Set the Test Digital Gains

In [ ]:
# Apply the inverse bin map when setting the digital gains
pc.i.chan[0].SCALER.USE_FLOAT_GAINS = 1
pc.i.chan[0].SCALER.set_gain_table(test_gains_flat[0, pc.inv_bin_map], bank=0, gain_timestamp=None, log_gain=None)

### Check the upper 16+16 bits of the 48+48-bit value after applying digital gains to the FFT data

In [ ]:
d_post_gain_flat = pc.read_post_gain_frames(n_frames = 2, 
                                            data_source = 'a', 
                                            fft_bypass = True, 
                                            a = 1
                                            )

In [ ]:
d_post_gain_flat.shape

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.title('post-gain upper 16+16 bits, flat')
plt.plot(pc.f, d_post_gain_flat[0, 0].real)
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.plot(pc.f, d_post_gain_flat[0, 0].imag)
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, np.abs(d_post_gain_flat[0, 0]), label = 'measured')
plt.hlines(test_target*2**12, 0, 1600, color = 'red', linestyle = '--', label = 'expected')
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

### Capture Scaler Data

In [ ]:
d_sclr_test = pc.read_scaler_frames(n_frames = 2,
                                    data_source = 'a',
                                    fft_bypass = True,
                                    scaler_bypass = False,
                                    a = 1
                                    )

In [ ]:
d_sclr_test.shape

In [ ]:
d_sclr_test_mean_mag = np.mean(np.abs(d_sclr_test[0]), axis = 0)
d_sclr_test_mean_mag

### Plot the Scaler Magnitude and Histograms

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(pc.f, d_sclr_test_mean_mag, label = 'mean')
plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('scaler magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.legend()
plt.grid()
plt.tight_layout()

In [ ]:
c_test_re, bin_edges = np.histogram(d_sclr_test[0].real.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
c_test_im, bin_edges = np.histogram(d_sclr_test[0].imag.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
bin_cents = np.arange(-7, 7 + 1)

plt.figure(figsize = (12, 4))

plt.subplot(121)
plt.title('real')
plt.bar(bin_cents, c_test_re/np.sum(c_test_re))
plt.vlines(-8, 0, np.max(c_test_re/np.sum(c_test_re)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_test_re/np.sum(c_test_re)), color = 'black', linestyle = '--')
plt.scatter(1, 1, color = 'red', label = 'expected')
plt.xticks(bin_cents)
plt.ylabel('normalized counts')
plt.xlabel('scaler LSBs')
plt.grid()
plt.legend()

plt.subplot(122)
plt.title('imag')
plt.bar(bin_cents, c_test_im/np.sum(c_test_im))
plt.vlines(-8, 0, np.max(c_test_im/np.sum(c_test_im)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_test_im/np.sum(c_test_im)), color = 'black', linestyle = '--')
plt.scatter(1, 1, color = 'red')
plt.xticks(bin_cents)
plt.xlabel('scaler LSBs')
plt.grid()

plt.tight_layout()

## Test with Ramp Function

As with the flat function, the ramp function will have a deterministic output. Therefore, we can use the same target as with the flat function; however, we will now have a gain that varies in frequency, meaning that we will need to apply a bin map or an inverse bin map where appropriate to interpret the data correctly.

In [ ]:
ramp_half = np.arange(pc.M//2)
ramp = np.zeros(len(ramp_half)*2)
ramp[::2] = ramp_half[pc.inv_bin_map]
ramp[1::2] = -1*ramp_half[pc.inv_bin_map]

In [ ]:
plt.figure(figsize = (12, 6))
plt.subplot(211)
plt.title('simulated custom ramp function, interpreted as an FFT')
plt.plot(pc.f, ramp[::2][pc.bin_map])
plt.xlim(0, 1600)
plt.ylabel('real')
plt.grid()
plt.subplot(212)
plt.plot(pc.f, ramp[1::2][pc.bin_map])
plt.xlim(0, 1600)
plt.ylabel('imag')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.tight_layout()

In [ ]:
d_fft_ramp = pc.read_fft_frames(n_frames = 2, 
                                fft_bit_res = '32+32', 
                                data_source = 'buffer', 
                                fft_bypass = True, 
                                period = None,
                                buffer = ramp
                                )

In [ ]:
d_fft_ramp_mean_mag = np.mean(np.abs(d_fft_ramp), axis = 1)

In [ ]:
# Plot the flat data to verify that it's being set up and read correctly

plt.figure(figsize = (12, 8))

plt.subplot(311)
plt.title('test ramp spectrum')
plt.plot(pc.f, d_fft_ramp[0, 0].real, label = 'measured')
plt.plot(pc.f, ramp[::2][pc.bin_map]*2**18, color = 'red', linestyle = '--', label = 'expected')
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.subplot(312)
plt.plot(pc.f, d_fft_ramp[0, 0].imag)
plt.plot(pc.f, ramp[1::2][pc.bin_map]*2**18, color = 'red', linestyle = '--', label = 'expected')
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, d_fft_ramp_mean_mag[0])
plt.plot(pc.f, np.sqrt((ramp[::2][pc.bin_map]**2 + ramp[1::2][pc.bin_map]**2))*(2**18), color = 'red', linestyle = '--', label = 'expected')
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()

plt.tight_layout()

### Set the Test Gains for the Ramp Function

In [ ]:
test_gains_ramp = test_target_full/d_fft_ramp_mean_mag

In [ ]:
test_gains_ramp.shape

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(311)
plt.title('verification of gain computation, computed gains')
plt.plot(pc.f, d_fft_ramp_mean_mag[0])
plt.ylabel('|FFT|')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.semilogy(pc.f, test_gains_ramp[0])
plt.ylabel('gains')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f[1:], d_fft_ramp_mean_mag[0][1:]*test_gains_ramp[0][1:]/2**44, label = 'gain $\\times$ |FFT|')
plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('simulated |scaler|')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

In [ ]:
# Treat infinities
for n in range(pc.NPOLS):
    test_gains_ramp[n][np.where(test_gains_ramp[n] == np.inf)[0]] = 0

In [ ]:
test_gains_ramp

In [ ]:
# Apply the inverse bin map when setting the digital gains
test_gains_ramp_inv_bin_map = test_gains_ramp[..., pc.inv_bin_map]

In [ ]:
# Treat infinities
for n in range(pc.NPOLS):
    test_gains_ramp_inv_bin_map[n][np.where(test_gains_ramp_inv_bin_map[n] == np.inf)[0]] = 1

In [ ]:
# Write the gains to the Scaler memory bank
pc.i.chan[0].SCALER.set_gain_table(test_gains_ramp_inv_bin_map[0], bank=0, gain_timestamp=None, log_gain=None)

In [ ]:
# Read the gains from the Scaler memory bank

read_gains_ramp = np.array(pc.i.chan[0].SCALER.get_gain_table(bank = 0)) & 0xFFFF
read_gains_common = 2**pc.i.chan[0].SCALER.SHIFT_LEFT
read_gains_ramp_corrected = read_gains_common*((read_gains_ramp & 0x7FF) << (read_gains_ramp >> 11)).flatten()
# read_gains_ramp_corrected = read_gains_ramp_corrected[..., pc.bin_map] # remember to apply the bin map, since the gains are written in scrambled order!
read_gains_ramp_corrected

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(411)
plt.title('verification of gain computation, gains read from memory')
plt.plot(pc.f, d_fft_ramp_mean_mag[0])
plt.ylabel('|FFT|')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(412)
plt.semilogy(pc.f, test_gains_ramp[0], label = 'gains (infinite resolution)')
plt.semilogy(pc.f, read_gains_ramp_corrected[pc.bin_map], linestyle = '--', label = 'gains actual')
plt.ylabel('gains')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.subplot(413)
plt.semilogy(pc.f, np.abs(read_gains_ramp_corrected[pc.bin_map] - test_gains_ramp[0]))
plt.ylabel('$\Delta$gains')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(414)
plt.plot(pc.f[1:], (d_fft_ramp_mean_mag[0][1:]*test_gains_ramp[0, pc.bin_map][1:]/2**44) - test_target, label = 'gains (infinite resolution) $\\times$ |FFT|')
plt.plot(pc.f[1:], (d_fft_ramp_mean_mag[0][1:]*read_gains_ramp_corrected[pc.bin_map][1:]/2**44) - test_target, linestyle = 'dotted', label = 'gains actual $\\times$ |FFT|')
# plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('simulated \n|scaler| - target')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

### Verify the Post-Gain Upper 16+16 Bits

In [ ]:
d_post_gain_ramp = pc.read_post_gain_frames(n_frames = 2, 
                                            data_source = 'buffer', 
                                            fft_bypass = True, 
                                            buffer = ramp
                                            )

In [ ]:
plt.figure(figsize = (12, 6))

plt.subplot(311)
plt.title('post-gain upper 16+16 bits, ramp')
plt.plot(pc.f, d_post_gain_ramp[0, 0].real)
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.plot(pc.f, d_post_gain_ramp[0, 0].imag)
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, np.abs(d_post_gain_ramp[0, 0]), label = 'measured')
plt.hlines(test_target*2**12, 0, 1600, color = 'C1', linestyle = '--', label = 'expected')
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend()

plt.tight_layout()

### Capture Scaler Data, Ramp

In [ ]:
d_sclr_ramp = pc.read_scaler_frames(n_frames = 2,
                                    data_source = 'buffer',
                                    fft_bypass = True,
                                    scaler_bypass = False,
                                    buffer = ramp
                                    )

In [ ]:
d_sclr_ramp_mean_mag = np.mean(np.abs(d_sclr_ramp[0]), axis = 0)
d_sclr_ramp_mean_mag

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(pc.f, d_sclr_ramp_mean_mag[pc.bin_map], label = 'mean')
plt.hlines(test_target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('scaler magnitude')
plt.xlabel('frequency [MHz]')
# plt.xlim(0, 1600)
plt.legend()
plt.grid()
plt.tight_layout()

In [ ]:
c_ramp_re, bin_edges = np.histogram(d_sclr_ramp[0].real.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
c_ramp_im, bin_edges = np.histogram(d_sclr_ramp[0].imag.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
bin_cents = np.arange(-7, 7 + 1)

plt.figure(figsize = (12, 4))

plt.subplot(121)
plt.title('real')
plt.bar(bin_cents, c_ramp_re/np.sum(c_ramp_re))
plt.vlines(-8, 0, np.max(c_ramp_re/np.sum(c_ramp_re)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_ramp_re/np.sum(c_ramp_re)), color = 'black', linestyle = '--')
plt.scatter(1, 1, color = 'red', label = 'expected')
plt.xticks(bin_cents)
plt.ylabel('normalized counts')
plt.xlabel('scaler LSBs')
plt.grid()
plt.legend()

plt.subplot(122)
plt.title('imag')
plt.bar(bin_cents, c_ramp_im/np.sum(c_ramp_im))
plt.vlines(-8, 0, np.max(c_ramp_im/np.sum(c_ramp_im)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_ramp_im/np.sum(c_ramp_im)), color = 'black', linestyle = '--')
plt.scatter(-1, 1, color = 'red', label = 'expected')
plt.xticks(bin_cents)
plt.xlabel('scaler LSBs')
plt.grid()

plt.tight_layout()

## Test with Real Signal

In [ ]:
target = 1.5*np.sqrt(2) # default value borrowed from CHIME

In [ ]:
# I recognize this overwrites the previous d_fft and d_fft_mean_mag instances, but since our test signal is stable, it 
# doesn't really matter; moreover, we want to test the compute_gains() function to verify that it can perform in a 
# standalone manner.

d_fft, d_fft_mean_mag, gain_lin_actual, gain_log_actual, d_sclr = pc.compute_gains(target = target,
                                                                                   n_fft_frames = n_fft_frames,
                                                                                   bank = 0,
                                                                                   return_scaler_frames = True,
                                                                                   save_gains = False
                                                                                   )

# d_fft, d_fft_mean_mag, gain_theory = pc.compute_gains(target = target,
#                                                                                    n_fft_frames = n_fft_frames,
#                                                                                    bank = 0,
#                                                                                    return_scaler_frames = True,
#                                                                                    save_gains = False
#                                                                                    )

In [ ]:
gain = gain_lin_actual * 2**gain_log_actual

In [ ]:
sim_sclr_mag = np.mean(np.abs(np.clip(np.round(d_fft*gain[:, None, :]/2**44), -7, 7)), axis = 1)

In [ ]:
sim_sclr_mag.shape

In [ ]:
d_sclr.shape

In [ ]:
test_sclr_mag = np.mean(np.abs(d_sclr), axis = 1)

In [ ]:
plt.figure(figsize = (12, 8))
plt.suptitle(f'simulated scaler output, ADC1')

plt.subplot(311)
plt.semilogy(pc.f, d_fft_mean_mag[0])
plt.ylabel('|FFT|')
plt.grid()

plt.subplot(312)
plt.semilogy(pc.f, gain[0])
plt.ylabel('gains')
plt.grid()

plt.subplot(313)
plt.plot(pc.f, sim_sclr_mag[0], label = 'gain (actual) $\\times$ |FFT|')
plt.plot(pc.f, test_sclr_mag[0], label = 'test scaler data')
plt.hlines(target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.ylabel('|scaler|')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend(fontsize = 10)

plt.tight_layout()

### Post-Gain 16+16 Real Data

In [ ]:
d_post_gain = pc.read_post_gain_frames(n_frames = n_fft_frames,
                                       data_source = 'adc'
                                       )

In [ ]:
d_post_gain_mean_mag = np.mean(np.abs(d_post_gain), axis = 1)

In [ ]:
plt.figure(figsize = (12, 8))
plt.suptitle(f'post-gain output, ADC1')

plt.subplot(311)
plt.plot(pc.f, d_post_gain[0, 0].real)
plt.ylabel('real')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(312)
plt.plot(pc.f, d_post_gain[0, 0].imag, color = 'C1')
plt.ylabel('imag')
plt.xlim(0, 1600)
plt.grid()

plt.subplot(313)
plt.plot(pc.f, d_post_gain_mean_mag[0], label = 'measured')
plt.plot(pc.f, d_fft_mean_mag[0]*gain[0]/2**32, label = 'simulated, gains actual')
plt.hlines(target*2**12, 0, 1600, color = 'red', linestyle = '--', label = 'target')
plt.xlim(0, 1600)
plt.ylabel('magnitude')
plt.xlabel('frequency [MHz]')
plt.legend(fontsize = 10)
plt.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 10))
plt.suptitle('post-gain FFT magnitude')

for n in range(pc.NPOLS):
    plt.subplot(4, 2, n + 1)
    plt.title(f'ADC{n + 1}')
    plt.plot(pc.f, d_post_gain_mean_mag[n], label = 'measured')
    plt.plot(pc.f, d_fft_mean_mag[n]*gain[n]/2**32, label = 'simulated')
    plt.hlines(target*2**12, 0, 1600, color = 'red', linestyle = '--', label = 'target')
    # plt.hlines(0, 0, 1600, color = 'black', linestyle = '--')
    # plt.hlines(7*np.sqrt(2), 0, 1600, color = 'black', linestyle = '--')

    if n in [0, 2, 4, 6]:
        plt.ylabel('|post-gain FFT|')
    if n in [6, 7]:
        plt.xlabel('frequency [MHz]')

    if n == 0:
        plt.legend(fontsize = 10)
    
    # plt.xticks(np.arange(0, 1600 + 100, 100))
    plt.xlim(0, 1600)
    plt.grid()
plt.tight_layout()

In [ ]:
d_post_gain_quant = np.clip(np.round(d_post_gain[0]/2**12), -7, 7)
d_post_gain_quant

In [ ]:
d_post_gain_quant_sim = np.clip(np.round(d_fft[0]*gain[0]/2**44), -7, 7)
# d_post_gain_quant_sim = d_fft[0]*read_gains_corrected/2**44

In [ ]:
plt.figure(figsize = (12, 4))
plt.title('expected scaler output, using post-gain data')
plt.plot(pc.f, np.mean(np.abs(d_post_gain_quant), axis = 0), label = 'measured, post-gain')
plt.plot(pc.f, np.mean(np.abs(d_post_gain_quant_sim), axis = 0), label = 'simulated, gain actual')
plt.hlines(target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
# plt.hlines(0, 0, 1600, color = 'black', linestyle = '--')
# plt.hlines(7*np.sqrt(2), 0, 1600, color = 'black', linestyle = '--')
plt.ylabel('|scaler|')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.grid()
plt.legend(fontsize = 10)
plt.tight_layout()

### Capture Scaler Data

In [ ]:
n_sclr_frames = n_fft_frames

In [ ]:
d_sclr = pc.read_scaler_frames(n_frames = n_sclr_frames,
                               data_source = 'adc'
                               )

In [ ]:
d_sclr_mean_mag = np.mean(np.abs(d_sclr), axis = 1)

In [ ]:
plt.figure(figsize = (12, 10))
plt.suptitle('scaler magnitude')

for n in range(pc.NPOLS):
    plt.subplot(4, 2, n + 1)
    plt.title(f'scaler/target ratio: {np.round(np.mean(d_sclr_mean_mag)/target, 2)}, ADC{n + 1}')
    plt.plot(pc.f, d_sclr_mean_mag[n], label = 'measured')
    plt.plot(pc.f, sim_sclr_mag[n], label = 'simulated')
    plt.hlines(target, 0, 1600, color = 'red', linestyle = '--', label = 'target')
    # plt.hlines(0, 0, 1600, color = 'black', linestyle = '--')
    # plt.hlines(7*np.sqrt(2), 0, 1600, color = 'black', linestyle = '--')

    if n in [0, 2, 4, 6]:
        plt.ylabel('|scaler|')
    if n in [6, 7]:
        plt.xlabel('frequency [MHz]')

    if n == 0:
        plt.legend(fontsize = 10)
    
    # plt.xticks(np.arange(0, 1600 + 100, 100))
    plt.xlim(0, 1600)
    plt.grid()
plt.tight_layout()

### Generate Scaler Histograms

In [ ]:
d_sclr[0].shape

In [ ]:
c_re, bin_edges = np.histogram(d_sclr[0].real.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
c_im, bin_edges = np.histogram(d_sclr[0].imag.flatten(), bins = np.arange(-7.5, 7.5 + 1, 1))
bin_cents = np.arange(-7, 7 + 1)

In [ ]:
plt.figure(figsize = (12, 4))

plt.subplot(121)
plt.title('real')
plt.bar(bin_cents, c_re/np.sum(c_re))
plt.vlines(-8, 0, np.max(c_re/np.sum(c_re)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_re/np.sum(c_re)), color = 'black', linestyle = '--')
plt.xticks(bin_cents)
plt.ylabel('normalized counts')
plt.xlabel('scaler LSBs')
plt.grid()

plt.subplot(122)
plt.title('imag')
plt.bar(bin_cents, c_im/np.sum(c_im))
plt.vlines(-8, 0, np.max(c_im/np.sum(c_im)), color = 'black', linestyle = '--')
plt.vlines(8, 0, np.max(c_im/np.sum(c_im)), color = 'black', linestyle = '--')
plt.xticks(bin_cents)
plt.xlabel('scaler LSBs')
plt.grid()

plt.tight_layout()

### Convert Scaler Units to FFT Units

In [ ]:
scaler_to_fft = 2**44/gain 
d_sclr_fft_units = scaler_to_fft*d_sclr_mean_mag

In [ ]:
plt.figure(figsize = (12, 6))

# plt.subplot(211)
plt.title('comparison of FFT magnitude and Scaler magnitude')
plt.semilogy(pc.f, d_fft_mean_mag[0], label = 'FFT')
plt.semilogy(pc.f, d_sclr_fft_units[0], label = 'scaler')
plt.ylabel('magnitude [FFT units]')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()
plt.legend()

plt.tight_layout()

# Capture Correlated Data

## Read Correlated Data for a Ramp Spectrum

In [ ]:
# Change the digital gains back to those for the ramp spectrum

for n in range(pc.NPOLS):
    # Apply the inverse bin map when setting the digital gains
    pc.i.chan[n].SCALER.USE_FLOAT_GAINS = 1
    pc.i.chan[n].SCALER.set_gain_table(test_gains_ramp_inv_bin_map[n])

In [ ]:
read_gains_ramp_corrected

In [ ]:
# Read the gains from the Scaler memory bank

read_gains_ramp_corrected2 = np.zeros((pc.NPOLS, pc.NBINS))

for n in range(pc.NPOLS):
    read_gains_ramp2 = np.array(pc.i.chan[n].SCALER.get_gain_table(bank = 0)) & 0xFFFF
    read_gains_ramp_common2 = 2**pc.i.chan[n].SCALER.SHIFT_LEFT
    read_gains_ramp_corrected2[n] = read_gains_ramp_common2*((read_gains_ramp2 & 0x7FF) << (read_gains_ramp2 >> 11)).flatten()

read_gains_ramp_corrected2 = read_gains_ramp_corrected2[:, pc.bin_map]

In [ ]:
# Configure the correlator

autocorr_only = 0
no_accum = 0
n_firmware_frames = 16384

pc.configure_corr(n_firmware_frames = n_firmware_frames,
                  autocorr_only = autocorr_only,
                  no_accum = no_accum,
                  )

In [ ]:
# Set the integration time through the number of software frames to integrate

n_software_frames = 119
s_per_frame = 2**14*(1/(pc.fs*1e6)) # samples/frame * seconds/sample
total_integration_time = n_firmware_frames*n_software_frames*s_per_frame
print(f'Total integration time = {total_integration_time} s')

In [ ]:
# Capture correlated data!

number_of_results = 1

t1_ramp, d_corr_ramp, counts_ramp, sat_ramp = pc.read_corr_frames(data_source = 'buffer',
                                                                  fft_bypass = True,
                                                                  scaler_bypass = False,
                                                                  n_software_frames = n_software_frames,
                                                                  number_of_results = number_of_results,
                                                                  flush = True,
                                                                  corr_is_configured = pc.corr_is_configured,
                                                                  buffer = ramp
                                                                  )

In [ ]:
pc.corr.OVERRUN

In [ ]:
d_corr_ramp.shape

In [ ]:
d_corr_ramp

In [ ]:
counts_ramp

### To-Debug

I don't think the sat values are actually reflective of any saturations in the firmware accumulator. If the Scaler magnitude is high (for example, with a mean magnitude of 3.5 or greater), the firmware accumulator will saturate, and the output will be a flat line, but the sat array will return all 0+0j.

In [ ]:
sat_ramp

### Check Out Autocorrelations

In [ ]:
# Recall that the correlator is accumulating visibilities formed by multiplying together
# two SCALER frames, then accumulating them. In this case, we are taking the sum of powers,
# rather than the sum of magnitudes. So to make a fair comparison, we must do the same for 
# the SCALER data.

d_sclr_ramp_mean_psd = np.mean(np.abs(d_sclr_ramp)**2, axis = 1)

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(pc.f, np.abs(d_corr_ramp[0, :, 0]), label = 'measured')
plt.plot(pc.f, n_firmware_frames*n_software_frames*d_sclr_ramp_mean_psd[0], color = 'r', ls = '--', label = 'expected')
# plt.hlines((2**18)**2, 0, 1600)
plt.xlim(0, 1600)
plt.ylabel('raw correlator output')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()
plt.tight_layout()

In [ ]:
pc.generate_prod_map()

In [ ]:
counts_ramp_good = counts_ramp[0] == n_software_frames*len(pc.f)

if counts_ramp_good:
    d_corr_ramp_sclr_units = np.zeros(d_corr_ramp.shape, dtype = np.complex128)
    for n in range(pc.NCORR_PROD):
        d_corr_ramp_sclr_units[..., n] = d_corr_ramp[..., n]/(n_software_frames*n_firmware_frames)
else:
    print(counts_ramp)
    raise ValueError('Check your counts array. You may have dropped packets.')

In [ ]:
plt.figure(figsize = (12, 10))

for n in range(pc.NCORR_PROD):
    if pc.prod[n][0] == pc.prod[n][1]:
        plt.subplot(4, 2, pc.prod[n][0] + 1)
        plt.title(f'ADC{pc.prod[n][0] + 1}')
        plt.plot(pc.f, d_sclr_ramp_mean_psd[pc.prod[0][0]], label = 'scaler magnitude', color = 'C1') # all pols should be the same, so use pol 0
        plt.plot(pc.f, np.abs(d_corr_ramp_sclr_units[0, :, n]), label = 'corr magnitude', color = 'C2') # np.abs simply casts complex128 to float
        plt.xlim(0, 1600)

        if pc.prod[n][0] in [0, 2, 4, 6]:
            plt.ylabel('|scaler|$^2$')
        if pc.prod[n][0] in [6, 7]:
            plt.xlabel('frequency [MHz]')
        if n == 0:
            plt.legend(fontsize = 10)
            
        plt.grid()

plt.tight_layout()

In [ ]:
# Hack to avoid division by 0 errors

read_gains_ramp_corrected2[np.where(read_gains_ramp_corrected2 == 0.)] = np.inf

In [ ]:
d_corr_ramp_fft_units = np.zeros(d_corr_ramp.shape, dtype = np.complex128)
for n in range(pc.NCORR_PROD):
    p0 = pc.prod[n][0]
    p1 = pc.prod[n][1]
    d_corr_ramp_fft_units[..., n] = ((2**44)**2/(read_gains_ramp_corrected2[p0]*read_gains_ramp_corrected2[p1]))*d_corr_ramp_sclr_units[..., n]

In [ ]:
# As we did with the SCALER data, we will also need to sum the power of
# FFT frames to match what the correaltor is doing.

d_fft_ramp_mean_psd = np.mean(np.abs(d_fft_ramp)**2, axis = 1)

In [ ]:
d_sclr_ramp_mean_psd_fft_units = (2**44/read_gains_ramp_corrected2)**2*d_sclr_ramp_mean_psd

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(211)
plt.title('ADC1')
plt.semilogy(pc.f, d_fft_ramp_mean_psd[0], label = 'FFT')
plt.semilogy(pc.f, d_sclr_ramp_mean_psd_fft_units[0], label = 'scaler')
plt.semilogy(pc.f, d_corr_ramp_fft_units[0, :, 0], label = 'auto')
plt.ylabel('power [arb.]')
plt.xlim(0, 1600)
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()
plt.legend() # loc = 'lower right')

plt.subplot(212)
plt.plot(pc.f, 100*(d_fft_ramp_mean_psd[0] - d_corr_ramp_fft_units[0, :, 0])/d_corr_ramp_fft_units[0, :, 0], label = 'corr', color = 'C0')
plt.plot(pc.f, 100*(d_sclr_ramp_mean_psd_fft_units[0] - d_corr_ramp_fft_units[0, :, 0])/d_corr_ramp_fft_units[0, :, 0], label = 'scaler', color = 'C1')
plt.hlines(0, 0, 1600, color = 'black', linestyle = '--')
plt.ylabel('frac. deviation \nfrom auto [%]')
plt.xlabel('frequency [MHz]')
plt.ylim(-100, 100)
plt.xlim(0, 1600)
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 10))
plt.title('comparison of FFT, scaler, and autocorrelation spectra')

for n in range(pc.NCORR_PROD):
    if pc.prod[n][0] == pc.prod[n][1]:
        plt.subplot(4, 2, pc.prod[n][0] + 1)
        plt.title(f'ADC{pc.prod[n][0] + 1}')
        plt.semilogy(pc.f, d_fft_ramp_mean_psd[pc.prod[n][0]], label = 'FFT')
        plt.semilogy(pc.f, d_sclr_ramp_mean_psd_fft_units[pc.prod[n][0]], label = 'scaler')
        plt.semilogy(pc.f, d_corr_ramp_fft_units[0, :, n], label = 'auto')
        plt.xlim(0, 1600)
        # plt.xticks(np.arange(0, 1600 + 100, 100))
        plt.grid()

        if pc.prod[n][0] in [0, 2, 4, 6]:
            plt.ylabel('power [arb.]')
        if pc.prod[n][0] in [6, 7]:
            plt.xlabel('frequency [MHz]')
        if n == 0:
            plt.legend(fontsize = 10)

plt.tight_layout()

### Check Out Cross-Correlations

Note that because we are using the same signals for each channelizer, the cross-correlations should match exactly with the autocorrelations.

In [ ]:
a = -2 + 1j*-2
a*np.conj(a)

In [ ]:
fig, ax = plt.subplots(8, 8, figsize = (24, 20))

for n in range(pc.NCORR_PROD):
    p0 = pc.prod[n][0]
    p1 = pc.prod[n][1]
    ax[p0, p1].set_title(f'ADC{pc.prod[n][0]}-ADC{pc.prod[n][1]}')
    if p0 == p1:
        # If it's an autocorrelation, use semilogy
        ax[p0, p1].semilogy(pc.f, np.abs(d_corr_ramp_fft_units[0, :, n]), lw = 5, color = 'black', label = 'measured')
        ax[p0, p1].semilogy(pc.f, d_fft_ramp_mean_psd[pc.prod[n][0]], color = 'yellow', ls = '--', label = 'expected')
    else:
        # If it's a cross-correlation, zoom in on the y-axis a little
        ax[p0, p1].plot(pc.f, d_corr_ramp_fft_units[0, :, n].real, color = 'blue', lw = 5, label = 'real, meas')
        ax[p0, p1].plot(pc.f, d_corr_ramp_fft_units[0, :, n].imag, color = 'orange', lw = 5, label = 'imag, meas')
        ax[p0, p1].plot(pc.f, (d_fft_ramp[0, 0]*d_fft_ramp[0, 0].conj()).real, color = 'yellow', ls = '--', label = 'expected')
        ax[p0, p1].plot(pc.f, (d_fft_ramp[0, 0]*d_fft_ramp[0, 0].conj()).imag, color = 'yellow', ls = '--')
        # ax[p0, p1].set_ylim(-1e9, 1e9)

    if n == 0 or n == 1:
        ax[p0, p1].legend()

    ax[p0, p1].set_xticks([0, 400, 800, 1200, 1600])
    ax[p0, p1].set_xlim(0, 1600)
    ax[p0, p1].grid()

fig.tight_layout()

## Read Correlated Data for Real ADC Data

In [ ]:
# Change the digital gains back to those for the real ADC data

for n in range(pc.NPOLS):
    # Apply the inverse bin map when setting the digital gains
    pc.i.chan[n].SCALER.set_gain_table(gain[n, pc.inv_bin_map])

In [ ]:
# Read the gains from the Scaler memory bank

read_gains_corrected2 = np.zeros((pc.NPOLS, pc.NBINS))

for n in range(pc.NPOLS):
    read_gains2 = np.array(pc.i.chan[n].SCALER.get_gain_table(bank = 0)) & 0xFFFF
    read_gains_common2 = 2**pc.i.chan[n].SCALER.SHIFT_LEFT
    read_gains_corrected2[n] = read_gains_common2*((read_gains2 & 0x7FF) << (read_gains2 >> 11)).flatten()

read_gains_corrected2 = read_gains_corrected2[:, pc.bin_map]

In [ ]:
# Capture correlated data!

number_of_results = 1

t1, d_corr, counts, sat = pc.read_corr_frames(data_source = 'adc',
                                              fft_bypass = False,
                                              scaler_bypass = False,
                                              n_software_frames = n_software_frames,
                                              number_of_results = number_of_results,
                                              flush = True,
                                              corr_is_configured = pc.corr_is_configured,
                                              )

In [ ]:
d_corr.shape

In [ ]:
counts

### Check Out Autocorrelations

In [ ]:
# Recall that the correlator is accumulating visibilities formed by multiplying together
# two SCALER frames, then accumulating them. In this case, we are taking the sum of powers,
# rather than the sum of magnitudes. So to make a fair comparison, we must do the same for 
# the SCALER data.

d_sclr_mean_psd = np.mean(np.abs(d_sclr)**2, axis = 1)

In [ ]:
d_sclr_mean_psd

In [ ]:
plt.figure(figsize = (12, 4))
plt.plot(pc.f, n_firmware_frames*n_software_frames*d_sclr_mean_psd[0], color = 'C1', label = 'scaler')
plt.plot(pc.f, np.abs(d_corr[0, :, 0]), color = 'C2', label = 'correlator')
plt.xlim(0, 1600)
plt.ylabel('raw correlator output')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()
plt.tight_layout()

In [ ]:
counts_good = counts[0] == n_software_frames*len(pc.f)

if counts_good:
    d_corr_sclr_units = np.zeros(d_corr.shape, dtype = np.complex128)
    for n in range(pc.NCORR_PROD):
        d_corr_sclr_units[..., n] = d_corr[..., n]/(n_software_frames*n_firmware_frames)
else:
    print(counts)
    raise ValueError('Check your counts array. You may have dropped packets.')

In [ ]:
plt.figure(figsize = (12, 10))

for n in range(pc.NCORR_PROD):
    if pc.prod[n][0] == pc.prod[n][1]:
        plt.subplot(4, 2, pc.prod[n][0] + 1)
        plt.title(f'ADC{pc.prod[n][0] + 1}')
        plt.plot(pc.f, d_sclr_mean_psd[pc.prod[n][0]], label = 'scaler magnitude', color = 'C1')
        plt.plot(pc.f, np.abs(d_corr_sclr_units[0, :, n]), label = 'corr magnitude', color = 'C2') # np.abs simply casts complex128 to float
        plt.xlim(0, 1600)

        if pc.prod[n][0] in [0, 2, 4, 6]:
            plt.ylabel('|scaler|$^2$')
        if pc.prod[n][0] in [6, 7]:
            plt.xlabel('frequency [MHz]')
        if n == 0:
            plt.legend(fontsize = 10)
            
        plt.grid()

plt.tight_layout()

In [ ]:
d_corr_fft_units = np.zeros(d_corr.shape, dtype = np.complex128)
for n in range(pc.NCORR_PROD):
    p0 = pc.prod[n][0]
    p1 = pc.prod[n][1]
    d_corr_fft_units[..., n] = ((2**44)**2/(gain[p0]*gain[p1]))*d_corr_sclr_units[..., n]

In [ ]:
# As we did with the SCALER data, we will also need to sum the power of
# FFT frames to match what the correaltor is doing.

d_fft_mean_psd = np.mean(np.abs(d_fft)**2, axis = 1)

In [ ]:
d_sclr_mean_psd_fft_units = (2**44/gain)**2*d_sclr_mean_psd

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(211)
plt.title('ADC1')
plt.semilogy(pc.f, d_fft_mean_psd[0], label = 'FFT')
plt.semilogy(pc.f, d_sclr_mean_psd_fft_units[0], label = 'scaler')
plt.semilogy(pc.f, d_corr_fft_units[0, :, 0], label = 'auto')
plt.ylabel('power [arb.]')
plt.xlim(0, 1600)
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()
plt.legend() # loc = 'lower right')

plt.subplot(212)
plt.plot(pc.f, 100*(d_fft_mean_psd[0] - d_corr_fft_units[0, :, 0])/d_corr_fft_units[0, :, 0], label = 'corr', color = 'C0')
plt.plot(pc.f, 100*(d_sclr_mean_psd_fft_units[0] - d_corr_fft_units[0, :, 0])/d_corr_fft_units[0, :, 0], label = 'scaler', color = 'C1')
plt.hlines(0, 0, 1600, color = 'black', linestyle = '--')
plt.ylabel('frac. deviation \nfrom auto [%]')
plt.xlabel('frequency [MHz]')
plt.ylim(-100, 100)
plt.xlim(0, 1600)
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 8))

plt.subplot(211)
plt.title('autocorrelation, ADC1')
plt.semilogy(pc.f, d_corr_fft_units[0, :, 0])
plt.ylabel('power [arb.]')
plt.xlim(0, 1600)
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()

plt.subplot(212)
plt.plot(pc.f, np.angle(d_corr_fft_units[0, :, 0]), color = 'green')
plt.ylabel('phase [rad.]')
plt.xlabel('frequency [MHz]')
plt.xlim(0, 1600)
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.grid()

plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 10))
plt.title('comparison of FFT, scaler, and autocorrelation spectra')

for n in range(pc.NCORR_PROD):
    if pc.prod[n][0] == pc.prod[n][1]:
        plt.subplot(4, 2, pc.prod[n][0] + 1)
        plt.title(f'ADC{pc.prod[n][0]}')
        plt.semilogy(pc.f, d_fft_mean_psd[pc.prod[n][0]], label = 'FFT')
        plt.semilogy(pc.f, d_sclr_mean_psd_fft_units[pc.prod[n][0]], label = 'scaler')
        plt.semilogy(pc.f, d_corr_fft_units[0, :, n], label = 'auto')
        plt.xlim(0, 1600)
        # plt.xticks(np.arange(0, 1600 + 100, 100))
        plt.grid()

        if pc.prod[n][0] in [0, 2, 4, 6]:
            plt.ylabel('power [arb.]')
        if pc.prod[n][0] in [6, 7]:
            plt.xlabel('frequency [MHz]')
        if n == 0:
            plt.legend(fontsize = 10)

plt.tight_layout()

## Check Out Cross-Correlations

In [ ]:
d_fft.shape

In [ ]:
# Cross-correlate the raw FFT data so we can compare. I realize this re-creates the autocorrelations of the raw FFT data, so I should 
# just move this earlier.

d_fft_mean_corr = np.zeros((pc.NCORR_PROD, pc.NBINS), dtype = np.complex128)

for n in range(pc.NCORR_PROD):
    p0 = pc.prod[n][0]
    p1 = pc.prod[n][1]
    d_fft_mean_corr[n] = np.mean(d_fft[p0]*d_fft[p1], axis = 0)

In [ ]:
d_fft_mean_corr.shape

In [ ]:
fig, ax = plt.subplots(8, 8, figsize = (24, 20))

for n in range(pc.NCORR_PROD):
    p0 = pc.prod[n][0]
    p1 = pc.prod[n][1]
    ax[p0, p1].set_title(f'ADC{pc.prod[n][0]}-ADC{pc.prod[n][1]}')
    if p0 == p1:
        # If it's an autocorrelation, use semilogy
        ax[p0, p1].semilogy(pc.f, np.abs(d_corr_fft_units[0, :, n]), lw = 5, color = 'black', label = 'measured')
        ax[p0, p1].semilogy(pc.f, d_fft_mean_psd[pc.prod[n][0]], color = 'yellow', ls = '--', label = 'expected')
    else:
        # If it's a cross-correlation, zoom in on the y-axis a little
        ax[p0, p1].plot(pc.f, d_corr_fft_units[0, :, n].real, color = 'blue', lw = 5, label = 'real, meas')
        ax[p0, p1].plot(pc.f, d_corr_fft_units[0, :, n].imag, color = 'orange', lw = 5, label = 'imag, meas')
        ax[p0, p1].plot(pc.f, d_fft_mean_corr[n].real, color = 'yellow', ls = '--', label = 'expected')
        ax[p0, p1].plot(pc.f, d_fft_mean_corr[n].imag, color = 'yellow', ls = '--')
        # ax[p0, p1].set_ylim(-1e9, 1e9)

    if n == 0 or n == 1:
        ax[p0, p1].legend()

    ax[p0, p1].set_xticks([0, 400, 800, 1200, 1600])
    ax[p0, p1].set_xlim(0, 1600)
    ax[p0, p1].grid()

fig.tight_layout()

In [ ]:
prod_sel = 1

plt.figure(figsize = (12, 6))

plt.subplot(221)
plt.title('real')
plt.plot(pc.f, d_corr_fft_units[0, :, prod_sel].real)
# plt.ylim(-1e9, 1e9)
plt.xlim(0, 1600)
plt.grid()

plt.subplot(222)
plt.title('magnitude')
plt.semilogy(pc.f, np.abs(d_corr_fft_units[0, :, prod_sel]))
plt.xlim(0, 1600)
plt.grid()

plt.subplot(223)
plt.title('imaginary')
plt.plot(pc.f, d_corr_fft_units[0, :, prod_sel].imag)
plt.xlim(0, 1600)
plt.xlabel('frequency [MHz]')
plt.grid()

plt.subplot(224)
plt.title('phase')
plt.plot(pc.f, np.angle(d_corr_fft_units[0, :, prod_sel]))
plt.xlim(0, 1600)
plt.xlabel('frequency [MHz]')
plt.grid()

plt.tight_layout()

In [ ]:
prod_sel = 1

plt.figure(figsize = (12, 6))

plt.title('phase')
plt.plot(pc.f, np.angle(d_corr_fft_units[0, :, prod_sel]))
plt.xlim(0, 100)
plt.xlabel('frequency [MHz]')
plt.grid()

plt.tight_layout()